In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Silver_Layer_Full_Audit") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.1.0,org.apache.hadoop:hadoop-aws:3.3.2") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

## Đọc dữ liệu và kiểm tra số lượng

In [4]:
# Đọc dữ liệu từ cả 2 tầng để so sánh
df_bronze = spark.read.format("delta").load("s3a://bronze/electronics_events")
df_silver = spark.read.format("delta").load("s3a://silver/electronics_events")

bronze_cnt = df_bronze.count()
silver_cnt = df_silver.count()

print(f"SỐ LƯỢNG BẢN GHI:")
print(f"- Tầng Bronze (Thô): {bronze_cnt:,}")
print(f"- Tầng Silver (Sạch): {silver_cnt:,}")
print(f"- Số dòng trùng lặp bị loại bỏ: {bronze_cnt - silver_cnt:,}")
print(f"- Tỷ lệ giữ lại dữ liệu: {(silver_cnt/bronze_cnt)*100:.2f}%")

SỐ LƯỢNG BẢN GHI:
- Tầng Bronze (Thô): 885,129
- Tầng Silver (Sạch): 884,292
- Số dòng trùng lặp bị loại bỏ: 837
- Tỷ lệ giữ lại dữ liệu: 99.91%


## Kiểm chứng kết quả Nội suy

In [5]:
print("KIỂM TRA DỮ LIỆU THIẾU TẠI CÁC CỘT QUAN TRỌNG:")
# Kiểm tra các cột đã được xử lý (fixed)
null_stats = df_silver.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in ["category_fixed", "brand_fixed", "product_name"]
])
null_stats.show()

# Kiểm tra xem có bao nhiêu dòng phải dùng nhãn "Category_[ID]"
orphan_cat = df_silver.filter(F.col("category_fixed").contains("Category_")).count()
print(f"Số lượng bản ghi được phải dùng Category_code bằng mã ID: {orphan_cat:,}")

# Kiểm tra bao nhiêu dòng phải dùng nhãn "Generic_Brand"
generic_brand = df_silver.filter(F.col("brand_fixed") == "Generic_Brand").count()
print(f"Số lượng bản ghi được gán nhãn Generic_Brand: {generic_brand:,}")

KIỂM TRA DỮ LIỆU THIẾU TẠI CÁC CỘT QUAN TRỌNG:
+--------------+-----------+------------+
|category_fixed|brand_fixed|product_name|
+--------------+-----------+------------+
|             0|          0|           0|
+--------------+-----------+------------+

Số lượng bản ghi được phải dùng Category_code bằng mã ID: 236,017
Số lượng bản ghi được gán nhãn Generic_Brand: 212,196


## Kiểm tra tính logic của Enrichment Check

In [6]:
print("MẪU DỮ LIỆU SAU KHI Enrichment Check (PRODUCT NAME):")
# Xem thử cách hệ thống ghép tên sản phẩm
df_silver.select("product_id", "brand_fixed", "category_fixed", "product_name") \
    .distinct() \
    .show(10, truncate=False)

MẪU DỮ LIỆU SAU KHI Enrichment Check (PRODUCT NAME):
+----------+-----------+--------------------------------+--------------------------------------------------+
|product_id|brand_fixed|category_fixed                  |product_name                                      |
+----------+-----------+--------------------------------+--------------------------------------------------+
|3828746   |eva        |electronics.audio.acoustic      |eva electronics.audio.acoustic #3828746           |
|139017    |deepcool   |computers.components.cooler     |deepcool computers.components.cooler #139017      |
|261923    |panasonic  |Category_2144415922939494519    |panasonic Category_2144415922939494519 #261923    |
|180228    |zalman     |computers.components.cooler     |zalman computers.components.cooler #180228        |
|3758770   |asus       |computers.components.motherboard|asus computers.components.motherboard #3758770    |
|1850104   |gigabyte   |computers.components.motherboard|gigabyte computers

## Phân tích hành vi người dùng

In [7]:
print("PHÂN BỔ LOẠI SỰ KIỆN TẠI TẦNG SILVER:")
df_silver.groupBy("event_type").count().orderBy(F.desc("count")).show()

# Kiểm tra xem 1 user trung bình thực hiện bao nhiêu hành vi
user_activity = df_silver.groupBy("user_id").count()
print(f"Số lượng User duy nhất: {user_activity.count():,}")
print("Thống kê số lượng hành vi mỗi User:")
user_activity.select("count").summary("mean", "min", "25%", "50%", "75%", "max").show()

PHÂN BỔ LOẠI SỰ KIỆN TẠI TẦNG SILVER:
+----------+------+
|event_type| count|
+----------+------+
|      view|792917|
|      cart| 54032|
|  purchase| 37343|
+----------+------+

Số lượng User duy nhất: 407,283
Thống kê số lượng hành vi mỗi User:
+-------+------------------+
|summary|             count|
+-------+------------------+
|   mean|2.1711979139811874|
|    min|                 1|
|    25%|                 1|
|    50%|                 1|
|    75%|                 2|
|    max|               572|
+-------+------------------+



## Kiểm tra Phân vùng vật lý

In [8]:
print("DANH SÁCH CÁC NGÀY SỰ KIỆN (EVENT_DATE PARTITIONS):")
# Kiểm tra xem dữ liệu có được chia folder đúng theo ngày khách mua hàng không
partitions = df_silver.select("event_date").distinct().orderBy("event_date")
print(f"Tổng số ngày (phân vùng): {partitions.count()}")
partitions.show(10)

DANH SÁCH CÁC NGÀY SỰ KIỆN (EVENT_DATE PARTITIONS):
Tổng số ngày (phân vùng): 158
+----------+
|event_date|
+----------+
|2020-09-24|
|2020-09-25|
|2020-09-26|
|2020-09-27|
|2020-09-28|
|2020-09-29|
|2020-09-30|
|2020-10-01|
|2020-10-02|
|2020-10-03|
+----------+
only showing top 10 rows

